# TabFM vs Meridian — official simulated data + deliverables matrix

> **Disclaimer:** All values below use **Google Meridian simulated/demo data**
> (and optional legacy dummy files). They are **not** estimates of real campaign
> performance and must not be used for budgeting or business decisions.

**Question this notebook answers:** Can TabFM output the **same deliverables**
as Meridian (contribution, ROI, response curves, budget optimization, geo insights)?

**Short answer:** TabFM can do **predictive KPI** scoring. It is **not** an MMM.
ROI / response curves / budget optimization are **No**. Contribution / geo are
**Partial** hacky proxies only — clearly labeled as **not Meridian-equivalent**.

**TabFM weight license:** `tabfm-non-commercial-v1.0` (non-commercial / non-production).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "mmm_compare").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from mmm_compare.compare import print_deliverables_matrix, print_table, run_comparison
from mmm_compare.data import DEFAULT_RUNTIME_DATASET, PREFERRED_OFFICIAL, load_mmm_dataset
from mmm_compare.deliverables import deliverables_dataframe, capability_summary

RESULTS = ROOT / "results"
# Runtime default is national (CPU-friendly). Preferred official file is geo.
DATASET = DEFAULT_RUNTIME_DATASET  # 'national' | 'geo' | 'geo-agg' | 'legacy'
DRY_RUN = True  # set False for real TabFM weights + Meridian MCMC

print("Preferred official:", PREFERRED_OFFICIAL)
print("Runtime dataset:", DATASET)
print("DRY_RUN:", DRY_RUN)

## 1) Deliverables matrix (honest gap analysis)

In [ ]:
caps = capability_summary()
print(caps["headline"])
print("Counts:", caps["counts"])
matrix = deliverables_dataframe()
display(matrix)

## 2) Load official Meridian simulated data

In [ ]:
data = load_mmm_dataset(dataset=DATASET, max_context_rows=100)
print("Source:", data.source_path)
print("KPI:", data.kpi_col, "| channels:", data.channel_keys)
print("Rows:", len(data.frame), "| train/test:", len(data.train_idx), len(data.test_idx))
print("Notes:", data.notes)
display(data.frame.head())

## 3) Run comparison (predictive metrics + Meridian deliverable tables)

In [ ]:
table, results, payload = run_comparison(
    dataset=DATASET,
    dry_run=DRY_RUN,
    prefer_real_meridian=True,
    results_dir=RESULTS,
)
print_table(table)
display(table)
print("Disclaimer:", payload["disclaimer"])
print("TabFM mode:", results["tabfm"].mode)
print("Meridian mode:", results["meridian"].mode)

## 4) Meridian deliverable artifacts (when real Meridian ran)

These come from Meridian `Analyzer` (contribution, ROI, summary metrics, response-curve preview).
TabFM does **not** produce Meridian-equivalent versions of these.

In [ ]:
mer = payload["deliverable_tables"]["meridian"]
tab = payload["deliverable_tables"]["tabfm"]

print("=== Meridian predictive_accuracy ===")
display(pd.DataFrame(mer.get("predictive_accuracy") or []))

print("=== Meridian channel contribution (incremental) ===")
display(pd.DataFrame(
    [{"channel": k, "incremental_outcome": v} for k, v in (mer.get("channel_contribution") or {}).items()]
))

print("=== Meridian ROI by channel ===")
display(pd.DataFrame(
    [{"channel": k, "roi": v} for k, v in (mer.get("roi_by_channel") or {}).items()]
))

print("=== Meridian summary_metrics (preview) ===")
sm = mer.get("summary_metrics_preview")
display(pd.DataFrame(sm) if isinstance(sm, list) else sm)

print("=== Response curves / budget opt ===")
print("response_curves:", type(mer.get("response_curves")),
      "| budget_optimization:", mer.get("budget_optimization"))

print("\n=== TabFM side (honest) ===")
print(tab.get("honesty"))
print("Ablation contribution proxy (NOT Meridian-equivalent):")
abl = (tab.get("channel_contribution_ablation_proxy") or {}).get("values") or {}
display(pd.DataFrame([{"channel": k, "ablation_proxy": v} for k, v in abl.items()]))

In [ ]:
import matplotlib.pyplot as plt

y = results["tabfm"].y_pred_test
n = min(len(y), len(results["meridian"].y_pred_test), 60)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(n), results["tabfm"].y_pred_test[:n], marker="s", label=f"TabFM ({results['tabfm'].mode})")
ax.plot(range(n), results["meridian"].y_pred_test[:n], marker="^", label=f"Meridian ({results['meridian'].mode})")
if len(data.y_test) == len(results["tabfm"].y_pred_test):
    ax.plot(range(n), data.y_test.to_numpy()[:n], marker="o", alpha=0.7, label="actual (simulated)")
ax.set_title("Holdout predictions (simulated Meridian data only)")
ax.set_ylabel(data.kpi_col)
ax.legend()
fig.tight_layout()
plt.show()

## How to run with real backends

```bash
bash scripts/install_deps.sh
INSTALL_MERIDIAN=1 bash scripts/install_deps.sh
python scripts/run_comparison.py -v                 # national default
python scripts/run_comparison.py --dataset geo -v   # may skip Meridian geo on CPU
```

Set `DRY_RUN = False` above after weights/Meridian are installed.